In [28]:
import duckdb
import pandas as pd
from pathlib import Path

con = duckdb.connect("../instacart.duckdb")


Query 01 - How many unique users are included in the dataset?

In [16]:
query_01 = Path("../sql/query_01.sql").read_text()

print(query_01)

print(con.execute(query_01).df())

SELECT COUNT(DISTINCT user_id) AS users_count
FROM orders;

   users_count
0       206209


Query 02 - How many orders have been placed?

In [17]:
query_02 = Path("../sql/query_02.sql").read_text()

print(query_02)

print(con.execute(query_02).df())

SELECT COUNT(DISTINCT order_id) as order_count
FROM orders;
   order_count
0      3421083


Query 03 - How many products belong to each department?

In [22]:
query_03 = Path("../sql/query_03.sql").read_text()

print(query_03)

print(con.execute(query_03).df())

SELECT d.department, COUNT(p.product_id) AS product_count
FROM products p
LEFT JOIN departments d ON p.department_id = d.department_id
GROUP BY d.department
ORDER BY COUNT(p.product_id) DESC;
         department  product_count
0     personal care           6563
1            snacks           6264
2            pantry           5371
3         beverages           4365
4            frozen           4007
5        dairy eggs           3449
6         household           3085
7      canned goods           2092
8   dry goods pasta           1858
9           produce           1684
10           bakery           1516
11             deli           1322
12          missing           1258
13    international           1139
14        breakfast           1115
15           babies           1081
16          alcohol           1054
17             pets            972
18     meat seafood            907
19            other            548
20             bulk             38


Query 04 - What are the 10 most frequently purchased products?

In [8]:
query_04 = Path("../sql/query_04.sql").read_text()

print(query_04)

print(con.execute(query_04).df())

SELECT p.product_name as name, count(*) as purchase_count
FROM order_products o
LEFT JOIN products p ON o.product_id = p.product_id
GROUP BY p.product_name
ORDER BY purchase_count DESC
LIMIT 10;
                     name  purchase_count
0                  Banana          491291
1  Bag of Organic Bananas          394930
2    Organic Strawberries          275577
3    Organic Baby Spinach          251705
4    Organic Hass Avocado          220877
5         Organic Avocado          184224
6             Large Lemon          160792
7            Strawberries          149445
8                   Limes          146660
9      Organic Whole Milk          142813


Query 05 - What are the 10 least frequently purchased products (with at least one purchase)?

In [25]:
query_05 = Path("../sql/query_05.sql").read_text()

print(query_05)

print(con.execute(query_05).df())

SELECT p.product_name as name, count(*) as purchase_count
FROM order_products o
LEFT JOIN products p ON o.product_id = p.product_id
GROUP BY p.product_id, p.product_name
HAVING purchase_count >= 1
ORDER BY purchase_count ASC
LIMIT 10;
                           name  purchase_count
0         Strawberry Energy Gel               1
1  Pasta Shapes In Tomato Sauce               1
2          Yellow Fish Breading               1
3     Orangemint Flavored Water               1
4                 Brut Prosecco               1
5                Original Lager               1
6             Jamaican Allspice               1
7             Salsa, Black Bean               1
8      Miss Treated Conditioner               1
9          Escapes Variety Pack               1


Query 06 - What is the average basket size?

In [18]:
query_06 = Path("../sql/query_06.sql").read_text()

print(query_06)

print(con.execute(query_06).df())

SELECT AVG(basket_size) as avg_basket_size
FROM(
    SELECT COUNT(product_id) as basket_size
    FROM order_products
    GROUP BY order_id
);
   avg_basket_size
0        10.107073


Query 07 - What is the largest basket in the dataset?

In [19]:
query_07 = Path("../sql/query_07.sql").read_text()

print(query_07)

print(con.execute(query_07).df())

SELECT MAX(basket_size) as max_basket_size
FROM(
    SELECT COUNT(product_id) as basket_size
    FROM order_products
    GROUP BY order_id
);
   max_basket_size
0              145


Query 08 - What is the distribution of basket sizes?

In [24]:
query_08 = Path("../sql/query_08.sql").read_text()

print(query_08)

print(con.execute(query_08).df())

WITH baskets AS (
    SELECT COUNT(product_id) as basket_size
    FROM order_products
    GROUP BY order_id
    )
SELECT basket_size, COUNT(*) as distribution
FROM baskets
GROUP BY basket_size
ORDER BY basket_size ASC;
     basket_size  distribution
0              1        163593
1              2        194361
2              3        215060
3              4        230299
4              5        237225
..           ...           ...
108          116             1
109          121             1
110          127             1
111          137             1
112          145             1

[113 rows x 2 columns]


Query 09 - Which departments have sold the largest number of products?

In [32]:
query_09 = Path("../sql/query_09.sql").read_text()

print(query_09)

print(con.execute(query_09).df())

SELECT p.department_id, d.department, COUNT(p.product_id) as product_count
FROM order_products o
LEFT JOIN products p ON o.product_id = p.product_id
LEFT JOIN departments d ON p.department_id = d.department_id
GROUP BY p.department_id, d.department
ORDER BY product_count DESC
LIMIT 5;
   department_id  department  product_count
0              4     produce        9888378
1             16  dairy eggs        5631067
2             19      snacks        3006412
3              7   beverages        2804175
4              1      frozen        2336858


Query 10 - Which aisles have the highest sales volume?

In [33]:
query_10 = Path("../sql/query_10.sql").read_text()

print(query_10)

print(con.execute(query_10).df())

SELECT 
a.aisle as aisle_name,
COUNT(o.product_id) as sales_volume
FROM order_products o
LEFT JOIN products p on o.product_id = p.product_id
LEFT JOIN aisles a on p.aisle_id = a.aisle_id
GROUP BY a.aisle
ORDER BY sales_volume DESC
LIMIT 5;
                   aisle_name  sales_volume
0                fresh fruits       3792661
1            fresh vegetables       3568630
2  packaged vegetables fruits       1843806
3                      yogurt       1507583
4             packaged cheese       1021462


Query 11 - Which products are most frequently added to the cart first?

In [45]:
query_11 = Path("../sql/query_11.sql").read_text()

print(query_11)

print(con.execute(query_11).df())

SELECT o.product_id, p.product_name, COUNT(*) as product_count
FROM order_products o
LEFT JOIN products p on o.product_id = p.product_id
WHERE o.add_to_cart_order = 1
GROUP BY o.product_id, p.product_name
ORDER BY product_count DESC
LIMIT 3;
   product_id            product_name  product_count
0       24852                  Banana         115521
1       13176  Bag of Organic Bananas          82877
2       27845      Organic Whole Milk          32071


Query 12 - Which products are most frequently added to the cart last?

In [46]:
query_12 = Path("../sql/query_12.sql").read_text()

print(query_12)

print(con.execute(query_12).df())


WITH last_item_in_order AS (
    SELECT 
    order_id, 
    MAX(add_to_cart_order) as last_item_added
    FROM order_products
    GROUP BY order_id
)
SELECT o.product_id as ID, p.product_name as product_name, COUNT(*) as product_count
FROM order_products o
JOIN last_item_in_order l ON o.order_id = l.order_id AND o.add_to_cart_order = l.last_item_added
LEFT JOIN products p ON o.product_id = p.product_id
GROUP BY o.product_id, p.product_name
ORDER BY product_count DESC
LIMIT 3;


      ID            product_name  product_count
0  24852                  Banana          31140
1  13176  Bag of Organic Bananas          30770
2  21137    Organic Strawberries          24359


Query 13 - Who are the top 20 users by number of orders?

In [ ]:
query_13 = Path("../sql/query_13.sql").read_text()

print(query_13)

print(con.execute(query_13).df())

Query 14 - Which department has the highest reorder rate?

In [ ]:
query_14 = Path("../sql/query_14.sql").read_text()

print(query_14)

print(con.execute(query_14).df())

Query 15 - Segment users based on the total number of orders they placed.

In [ ]:
query_15 = Path("../sql/query_15.sql").read_text()

print(query_15)

print(con.execute(query_15).df())

Query 16 - What percentage of purchased products are reordered?

In [ ]:
query_16 = Path("../sql/query_16.sql").read_text()

print(query_16)

print(con.execute(query_16).df())

Query 17 - Which aisles have the highest reorder rates?

In [ ]:
query_17 = Path("../sql/query_17.sql").read_text()

print(query_17)

print(con.execute(query_17).df())

Query 18 - What is the average basket size for each department?

In [ ]:
query_18 = Path("../sql/query_18.sql").read_text()

print(query_18)

print(con.execute(query_18).df())

Query 19 - Find the top 3 best-selling products within each department.

In [ ]:
query_19 = Path("../sql/query_19.sql").read_text()

print(query_19)

print(con.execute(query_19).df())

Query 20 - Rank all products by total number of purchases.

In [ ]:
query_20 = Path("../sql/query_20.sql").read_text()

print(query_20)

print(con.execute(query_20).df())

Query 21 - Which users belong to the top 10% by number of orders?

In [ ]:
query_21 = Path("../sql/query_21.sql").read_text()

print(query_21)

print(con.execute(query_21).df())

Query 22 - Assign a sequential order number to each user's purchases using window functions only.

In [ ]:
query_22 = Path("../sql/query_22.sql").read_text()

print(query_22)

print(con.execute(query_22).df())

Query 23 - Compare each basket size with the customer's previous basket.

In [ ]:
query_23 = Path("../sql/query_23.sql").read_text()

print(query_23)

print(con.execute(query_23).df())

Query 24 - Which users consistently increase the size of their baskets over time?

In [ ]:
query_24 = Path("../sql/query_24.sql").read_text()

print(query_24)

print(con.execute(query_24).df())

Query 25 - Find users who place orders regularly with less than 7 days between purchases.

In [ ]:
query_25 = Path("../sql/query_25.sql").read_text()

print(query_25)

print(con.execute(query_25).df())

Query 26 - Which products have been purchased by more than 10% of all users?

In [ ]:
query_26 = Path("../sql/query_26.sql").read_text()

print(query_26)

print(con.execute(query_26).df())

Query 27 - Which department generates the highest number of reordered products?

In [ ]:
query_27 = Path("../sql/query_27.sql").read_text()

print(query_27)

print(con.execute(query_27).df())

Query 28 - Find the product with the highest reorder rate within each department.

In [ ]:
query_28 = Path("../sql/query_28.sql").read_text()

print(query_28)

print(con.execute(query_28).df())

Query 29 - Which pair of products is purchased together most frequently?

In [ ]:
query_29 = Path("../sql/query_29.sql").read_text()

print(query_29)

print(con.execute(query_29).df())

Query 30 - Which combination of three products is purchased together most frequently?

In [ ]:
query_30 = Path("../sql/query_30.sql").read_text()

print(query_30)

print(con.execute(query_30).df())

Query 31 - Which products are most commonly purchased before another specific product?

In [ ]:
query_31 = Path("../sql/query_31.sql").read_text()

print(query_31)

print(con.execute(query_31).df())

Query 32 - Which products are most frequently purchased together with bananas?

In [ ]:
query_32 = Path("../sql/query_32.sql").read_text()

print(query_32)

print(con.execute(query_32).df())

Query 33 - Which user is the most loyal to a single department?

In [ ]:
query_33 = Path("../sql/query_33.sql").read_text()

print(query_33)

print(con.execute(query_33).df())

Query 34 - Which department has the greatest diversity of products purchased?

In [ ]:
query_34 = Path("../sql/query_34.sql").read_text()

print(query_34)

print(con.execute(query_34).df())

Query 35 - Which product has the highest reorder rate among products purchased at least 1,000 times?

In [ ]:
query_35 = Path("../sql/query_35.sql").read_text()

print(query_35)

print(con.execute(query_35).df())

Query 36 - For each department, calculate: total number of products, total number of purchases, reorder rate, and percentage share of total sales.

In [ ]:
query_36 = Path("../sql/query_36.sql").read_text()

print(query_36)

print(con.execute(query_36).df())

Query 37 - Find the top 5 best-selling products in each department, including ties.

In [ ]:
query_37 = Path("../sql/query_37.sql").read_text()

print(query_37)

print(con.execute(query_37).df())

Query 38 - For each user, determine: first order date, last order date, and number of days between the first and last order.

In [ ]:
query_38 = Path("../sql/query_38.sql").read_text()

print(query_38)

print(con.execute(query_38).df())

Query 39 - Calculate the median basket size.

In [ ]:
query_39 = Path("../sql/query_39.sql").read_text()

print(query_39)

print(con.execute(query_39).df())

Query 40 - Calculate the rolling average basket size for each user over time.

In [ ]:
query_40 = Path("../sql/query_40.sql").read_text()

print(query_40)

print(con.execute(query_40).df())

In [26]:
con.close()